# Analyse Exploratoire du WELFake Dataset
## Dataset externe pour test de generalisation hors domaine (OOD)

**Objectif** : Explorer le dataset WELFake (72 134 articles, fusion de 4 sources) pour comprendre sa structure et ses particularites avant de l'utiliser comme jeu de test hors domaine.

**Plan** :
1. Chargement des donnees
2. Analyse des labels de veracite
3. Qualite des donnees (NaN, doublons)
4. Analyse textuelle (corps + titres)
5. Analyse du vocabulaire par classe
6. Analyse specifique des titres
7. Comparaison avec LIAR et ISOT (domain shift)
8. Preprocessing et sauvegarde
9. Synthese

**Source** : Verma P. et al. (2021) — *WELFake: Word Embedding over Linguistic Features for Fake News Detection*, IEEE Transactions on Computational Social Systems. Fusion de 4 datasets : Kaggle, McIntire, Reuters, BuzzFeed Political.

In [16]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from collections import Counter
import re
import string
import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/externes/WELFake")
DATA_OUT = Path("../data/traitees")
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Palette de couleurs coherente
COLORS_BIN = ["#e74c3c", "#2ecc71"]
COLORS_BIN_MAP = {"Fake": "#e74c3c", "Real": "#2ecc71"}
COLORS_6 = ["#e74c3c", "#e67e22", "#f39c12", "#3498db", "#2ecc71", "#27ae60"]

print("Imports OK")

Imports OK


## 1. Chargement des donnees

WELFake est une **fusion de 4 datasets** :
- **Kaggle** (Fake and Real News Dataset, ~44K articles)
- **McIntire** (~6K articles)
- **Reuters** (~11K articles)
- **BuzzFeed Political** (~1K articles)

**Colonnes** : `title` (titre), `text` (corps de l'article), `label` (0=Fake, 1=Real)

> **Attention** : Certaines lignes ont des titres ou textes manquants (NaN) — c'est un artefact de la fusion.

In [17]:
df = pd.read_csv(DATA_DIR / "WELFake.csv")

# Ajouter label_name pour coherence avec les autres notebooks
df["label_name"] = df["label"].map({0: "Fake", 1: "Real"})

print(f"Nombre d'articles : {len(df):,}")
print(f"Colonnes : {list(df.columns)}")
print(f"\n=== Types ===")
print(df.dtypes)
print(f"\n=== Valeurs manquantes ===")
print(df.isnull().sum())
print(f"\n=== Doublons ===")
print(f"Doublons exacts : {df.duplicated().sum()}")
df.head()

Nombre d'articles : 72,134
Colonnes : ['title', 'text', 'label', 'label_name']

=== Types ===
title           str
text            str
label         int64
label_name      str
dtype: object

=== Valeurs manquantes ===
title         558
text           39
label           0
label_name      0
dtype: int64

=== Doublons ===
Doublons exacts : 8456


,title,text,label,label_name
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,Real
1,NaN,Did they post their votes for Hillary already?,1,Real
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,Real
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,Fake
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,Real


In [18]:
# Apercu statistique
df.describe(include="all")

,title,text,label,label_name
count,71576,72095,72134.000000,72134
unique,62347,62718,NaN,2
top,Factbox: Trump fills top jobs for his administ...,,NaN,Real
freq,14,738,NaN,37106
mean,NaN,NaN,0.514404,NaN
std,NaN,NaN,0.499796,NaN
min,NaN,NaN,0.000000,NaN
25%,NaN,NaN,0.000000,NaN
50%,NaN,NaN,1.000000,NaN
75%,NaN,NaN,1.000000,NaN


### Nettoyage initial (NaN et doublons)

WELFake est une fusion de 4 sources : il y a des **valeurs manquantes** (558 titres, 39 textes)
et des **doublons** (8 456). On nettoie avant analyse pour eviter les biais.

In [19]:
# --- Nettoyage initial ---
n_avant = len(df)

# Afficher les NaN par colonne
print("=== Valeurs manquantes ===")
print(df.isna().sum())

# Supprimer les lignes ou le texte (corps) est NaN
n_nan_text = df["text"].isna().sum()
df = df.dropna(subset=["text"])
print(f"
Lignes sans texte supprimees : {n_nan_text}")

# Remplir les titres manquants par chaine vide (on les garde)
n_nan_title = df["title"].isna().sum()
df["title"] = df["title"].fillna("")
print(f"Titres manquants remplaces par '' : {n_nan_title}")

# Supprimer les doublons exacts
n_dup = df.duplicated(subset=["text"]).sum()
df = df.drop_duplicates(subset=["text"])
print(f"Doublons (meme texte) supprimes : {n_dup}")

# Reset index
df = df.reset_index(drop=True)
print(f"
Dataset : {n_avant:,} -> {len(df):,} articles")
df.head()

SyntaxError: unterminated f-string literal (detected at line 11) (48968501.py, line 11)

## 2. Analyse des labels de veracite

Comme ISOT, WELFake est **binaire** (Fake/Real). Verifions l'equilibre des classes sur ce dataset plus gros.

In [ ]:
# Distribution des labels
label_counts = df["label_name"].value_counts()
print("Distribution des labels :")
for lab, n in label_counts.items():
    print(f"  {lab}: {n:>6,}  ({n/len(df)*100:.1f}%)")
print(f"\nRatio Fake/Real : {label_counts.get('Fake',0)/label_counts.get('Real',1):.3f}")

Distribution des labels :
  Real: 37,106  (51.4%)
  Fake: 35,028  (48.6%)

Ratio Fake/Real : 0.944


In [ ]:
# Visualisation : Distribution Fake vs Real
fig = px.pie(
    names=label_counts.index,
    values=label_counts.values,
    color=label_counts.index,
    color_discrete_map=COLORS_BIN_MAP,
    title="Distribution Fake / Real — WELFake (72 134 articles)",
    hole=0.4,
    template="plotly_white",
)
fig.update_traces(textinfo="label+percent+value", textfont_size=14)
fig.update_layout(height=420, font=dict(size=13))
fig.show()

## 3. Qualite des donnees (NaN, doublons)

WELFake etant une **fusion de 4 sources**, il est important d'analyser la qualite des donnees : valeurs manquantes, doublons, et coherence par label.

In [ ]:
# Valeurs manquantes par colonne et par label
print("=== Valeurs manquantes par label ===")
for lab in ["Fake", "Real"]:
    sub = df[df["label_name"] == lab]
    print(f"\n  {lab} ({len(sub):,} lignes) :")
    print(f"    title NaN : {sub['title'].isnull().sum():>5,}  ({sub['title'].isnull().mean()*100:.1f}%)")
    print(f"    text NaN  : {sub['text'].isnull().sum():>5,}  ({sub['text'].isnull().mean()*100:.1f}%)")

# Doublons
print(f"\n=== Doublons ===")
print(f"Doublons (text exact) : {df['text'].dropna().duplicated().sum():,}")
print(f"Doublons (title exact) : {df['title'].dropna().duplicated().sum():,}")

=== Valeurs manquantes par label ===

  Fake (35,028 lignes) :
    title NaN :     0  (0.0%)
    text NaN  :     0  (0.0%)

  Real (37,106 lignes) :
    title NaN :   558  (1.5%)
    text NaN  :    39  (0.1%)

=== Doublons ===
Doublons (text exact) : 9,377
Doublons (title exact) : 9,229


## 4. Analyse textuelle (corps + titres)

On analyse la **longueur des textes** (corps et titres) pour chaque classe. WELFake a l'avantage d'avoir des titres, ce qui offre une dimension supplementaire.

In [ ]:
# Features textuelles
df["n_words_text"] = df["text"].fillna("").apply(lambda x: len(x.split()) if x else 0)
df["n_words_title"] = df["title"].fillna("").apply(lambda x: len(x.split()) if x else 0)
df["n_chars_text"] = df["text"].fillna("").str.len()
df["n_sentences"] = df["text"].fillna("").str.count(r'[.!?]+')

print("=== Statistiques — corps de texte ===")
print(df.groupby("label_name")["n_words_text"].describe().round(1))
print("\n=== Statistiques — titres ===")
print(df.groupby("label_name")["n_words_title"].describe().round(1))

=== Statistiques — corps de texte ===
              count   mean    std  min    25%    50%    75%      max
label_name                                                          
Fake        35028.0  577.6  563.1  0.0  241.0  427.0  783.0  14641.0
Real        37106.0  505.6  677.1  0.0  217.0  369.0  568.0  24234.0

=== Statistiques — titres ===
              count  mean  std  min   25%   50%   75%   max
label_name                                                 
Fake        35028.0  11.0  2.9  2.0   9.0  11.0  13.0  30.0
Real        37106.0  13.2  5.0  0.0  10.0  13.0  16.0  72.0


In [ ]:
# Distribution du nombre de mots (corps) par label
fig = px.histogram(
    df[df["n_words_text"] > 0], x="n_words_text", color="label_name",
    color_discrete_map=COLORS_BIN_MAP,
    nbins=80, barmode="overlay", opacity=0.65,
    title="Distribution du nombre de mots (corps) par article — WELFake",
    labels={"n_words_text": "Nombre de mots", "label_name": "Label"},
    template="plotly_white",
)
fig.update_layout(height=420, font=dict(size=13))
fig.show()

In [ ]:
# Box plot comparatif : corps et titres
fig = make_subplots(rows=1, cols=2, subplot_titles=("Corps de texte", "Titres"))

for label, color in COLORS_BIN_MAP.items():
    sub = df[df["label_name"] == label]
    fig.add_trace(
        go.Box(y=sub["n_words_text"], name=label, marker_color=color, showlegend=True),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(y=sub["n_words_title"], name=label, marker_color=color, showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    title="Nombre de mots — Fake vs Real (WELFake)",
    height=450, template="plotly_white", font=dict(size=12)
)
fig.update_yaxes(title_text="Nombre de mots", row=1, col=1)
fig.update_yaxes(title_text="Nombre de mots", row=1, col=2)
fig.show()

## 5. Analyse du vocabulaire par classe

Comparons les **mots les plus frequents** dans les articles faux vs reels. On utilise un echantillon de 30K articles pour la performance.

In [ ]:
STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "was", "are", "were", "be", "been",
    "being", "have", "has", "had", "do", "does", "did", "will", "would",
    "could", "should", "may", "might", "shall", "can", "not", "no", "that",
    "this", "it", "he", "she", "they", "we", "you", "i", "his", "her",
    "their", "its", "my", "your", "our", "as", "if", "than", "so", "up",
    "out", "about", "who", "what", "which", "when", "where", "how", "all",
    "each", "more", "also", "after", "before", "said", "s", "t", "nan",
    "new", "one", "two", "just", "over", "into", "only", "other", "some",
}

def get_top_words(texts, n=20):
    counter = Counter()
    for text in texts:
        tokens = str(text).lower().translate(str.maketrans("", "", string.punctuation)).split()
        counter.update(w for w in tokens if w not in STOP_WORDS and len(w) > 2)
    return counter.most_common(n)

# Echantillon pour performance (72K articles = lourd)
sample = df.sample(n=min(30000, len(df)), random_state=42)
top_fake = get_top_words(sample[sample["label"] == 0]["text"].dropna(), 20)
top_real = get_top_words(sample[sample["label"] == 1]["text"].dropna(), 20)

print("=== Top 20 mots — FAKE ===")
for w, c in top_fake:
    print(f"  {w:20s} {c:>8,}")
print("\n=== Top 20 mots — REAL ===")
for w, c in top_real:
    print(f"  {w:20s} {c:>8,}")

=== Top 20 mots — FAKE ===
  trump                  34,552
  president              19,642
  people                 16,439
  state                  14,598
  states                 13,215
  there                  12,041
  government             12,023
  reuters                11,928
  house                  11,830
  united                 11,705
  last                   10,934
  him                    10,898
  republican             10,639
  like                   10,445
  told                   10,110
  them                    9,788
  clinton                 9,786
  first                   9,533
  campaign                9,389
  year                    9,260

=== Top 20 mots — REAL ===
  trump                  39,430
  people                 19,370
  clinton                16,635
  president              14,655
  there                  14,444
  like                   12,629
  hillary                11,782
  them                   11,125
  obama                  10,860
  because        

In [ ]:
# Visualisation comparative
fig = make_subplots(rows=1, cols=2, subplot_titles=("Top 20 mots — Fake", "Top 20 mots — Real"))

for i, (top, color, name) in enumerate([
    (top_fake, "#e74c3c", "Fake"),
    (top_real, "#2ecc71", "Real"),
]):
    words, counts = zip(*reversed(top))
    fig.add_trace(
        go.Bar(y=list(words), x=list(counts), orientation="h",
               marker_color=color, name=name),
        row=1, col=i+1
    )

fig.update_layout(
    title="Vocabulaire le plus frequent par classe — WELFake",
    height=550, template="plotly_white", showlegend=False, font=dict(size=12)
)
fig.show()

## 6. Analyse specifique des titres

Les titres sont souvent le **premier signal** de fake news (clickbait, sensationnalisme). Analysons leur longueur et vocabulaire specifique.

In [ ]:
# Distribution longueur des titres
fig = px.histogram(
    df[df["title"].notna()], x="n_words_title", color="label_name",
    color_discrete_map=COLORS_BIN_MAP,
    nbins=50, barmode="overlay", opacity=0.65,
    title="Distribution du nombre de mots dans les titres — WELFake",
    labels={"n_words_title": "Nombre de mots (titre)", "label_name": "Label"},
    template="plotly_white",
)
fig.update_layout(height=420, font=dict(size=13))
fig.show()

In [ ]:
# Top mots dans les titres
top_fake_titles = get_top_words(sample[sample["label"] == 0]["title"].dropna(), 15)
top_real_titles = get_top_words(sample[sample["label"] == 1]["title"].dropna(), 15)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Titres Fake — Top 15", "Titres Real — Top 15"))
for i, (top, color) in enumerate([(top_fake_titles, "#e74c3c"), (top_real_titles, "#2ecc71")]):
    words, counts = zip(*reversed(top))
    fig.add_trace(
        go.Bar(y=list(words), x=list(counts), orientation="h", marker_color=color),
        row=1, col=i+1
    )
fig.update_layout(
    title="Mots les plus frequents dans les titres — WELFake",
    height=480, template="plotly_white", showlegend=False, font=dict(size=12)
)
fig.show()

## 7. Comparaison avec LIAR et ISOT (domain shift)

Comparons les **3 datasets** pour mesurer le domain shift. Cette comparaison est essentielle pour comprendre les performances OOD attendues.

In [ ]:
# Tableau comparatif des 3 datasets
liar_path = Path("../data/traitees/liar_complet.parquet")
isot_path = Path("../data/traitees/isot_clean.parquet")

rows = []

# WELFake
wf_words = df[df["n_words_text"] > 0]["n_words_text"]
rows.append({
    "Dataset": "WELFake", "Nb articles": f"{len(df):,}",
    "Mots (mediane)": f"{wf_words.median():.0f}",
    "Mots (moyenne)": f"{wf_words.mean():.0f}",
    "Type contenu": "Articles + titres",
    "Sources": "4 sources fusionnees",
})

# ISOT
if isot_path.exists():
    df_isot = pd.read_parquet(isot_path)
    rows.append({
        "Dataset": "ISOT", "Nb articles": f"{len(df_isot):,}",
        "Mots (mediane)": f"{df_isot['n_words'].median():.0f}",
        "Mots (moyenne)": f"{df_isot['n_words'].mean():.0f}",
        "Type contenu": "Articles complets",
        "Sources": "Reuters + non fiable",
    })

# LIAR
if liar_path.exists():
    df_liar = pd.read_parquet(liar_path)
    liar_words = df_liar["statement"].astype(str).str.split().str.len()
    rows.append({
        "Dataset": "LIAR", "Nb articles": f"{len(df_liar):,}",
        "Mots (mediane)": f"{liar_words.median():.0f}",
        "Mots (moyenne)": f"{liar_words.mean():.0f}",
        "Type contenu": "Declarations courtes",
        "Sources": "PolitiFact",
    })

print("=== Comparaison des 3 datasets ===")
print(pd.DataFrame(rows).to_string(index=False))

=== Comparaison des 3 datasets ===
Dataset Nb articles Mots (mediane) Mots (moyenne)         Type contenu              Sources
WELFake      72,134            401            546    Articles + titres 4 sources fusionnees
   ISOT      34,100            373            410    Articles complets Reuters + non fiable
   LIAR      12,791             17             18 Declarations courtes           PolitiFact


## 8. Preprocessing et sauvegarde

Nettoyage minimal puis sauvegarde en Parquet. On combine titre + texte en un champ unique `full_text` pour l'evaluation.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Nettoyer texte et titre
df["text_clean"] = df["text"].fillna("").apply(clean_text)
df["title_clean"] = df["title"].fillna("").apply(clean_text)

# Combiner titre + texte pour un champ unique
df["full_text"] = (df["title_clean"] + " " + df["text_clean"]).str.strip()

# Suppression textes vides et doublons
n_before = len(df)
df = df[df["full_text"].str.len() > 10].drop_duplicates(subset=["full_text"])
print(f"Nettoyage : {n_before:,} -> {len(df):,} articles ({n_before - len(df)} supprimes)")

# Recalculer n_words
df["n_words"] = df["full_text"].str.split().str.len()

# Sauvegarde
out_path = DATA_OUT / "welfake_clean.parquet"
cols_keep = ["title_clean", "text_clean", "full_text", "label", "label_name", "n_words"]
df[cols_keep].to_parquet(out_path, index=False)
print(f"Sauvegarde -> {out_path}")
print(f"Shape finale : {df[cols_keep].shape}")

Nettoyage : 72,134 -> 63,655 articles (8479 supprimes)
Sauvegarde -> ..\data\traitees\welfake_clean.parquet
Shape finale : (63655, 6)


## 9. Synthese de l'EDA

**Observations cles** :

1. **Taille** : 72 134 articles — le plus gros dataset externe (~5.6x LIAR)
2. **Equilibre** : Quasi-equilibre, pas besoin de SMOTE pour l'evaluation
3. **Titres** : Dimension supplementaire absente de LIAR et ISOT — utile pour detecter le clickbait
4. **Qualite** : Valeurs manquantes (title/text NaN) — artefact de la fusion de 4 sources
5. **Diversite** : 4 sources differentes = diversite stylistique realiste pour un test OOD
6. **Domain shift** : Majeur par rapport a LIAR — articles longs multi-sources vs declarations courtes

**Impact pour la modelisation** :
- WELFake est le test OOD le plus exigeant (volume + diversite)
- La fusion de 4 sources simule un scenario realiste de detection en conditions reelles
- Le champ `full_text` (titre + corps) sera utilise pour l'evaluation OOD des modeles entraines sur LIAR